# PEBBLE H1/H2 on Colab (A100)

**Study:** when PEBBLE true return drops under paper ablations, is it reward-model ranking (**H1**) or SAC amplification (**H2**)?  
**Pre-registration:** `experiments/pebble_reward_vs_rl/CLAIM.md`  
**Not B-Pref** — Oracle teacher only (Lee et al., ICML 2021).

## Setup (do this first)
1. **Runtime → Change runtime type → GPU → A100** (Colab Pro / Pro+; not guaranteed on free).
2. Run cells top-to-bottom.
3. Keep **Drive mounted** so long suites survive disconnects.

## Status vocabulary
| Stage | This notebook |
|-------|----------------|
| Smoke | wiring only — not evidence |
| Diagnostic (5×4×100k) | tentative patterns only |
| Paper-scale (10×4×≥500k) | claim gate (CLAIM.md §6) |

In [ ]:
# Cell 1 — GPU check (prefer A100)
import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime → Change runtime type → GPU → A100 (or any NVIDIA GPU)."
)
name = torch.cuda.get_device_name(0)
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name} ({mem_gb:.1f} GB)")

is_a100 = "A100" in name.upper()
if not is_a100:
    print(
        "WARNING: This is not an A100. Suites still run on T4/L4/V100, but slower.\n"
        "Reconnect / change runtime until you get A100 if that is required."
    )
else:
    print("A100 detected — good for diagnostic + paper-scale.")

In [ ]:
# Cell 2 — Drive persistence (recommended for multi-hour suites)
from pathlib import Path

USE_DRIVE = True  # set False only for short smoke tests
REPO_URL = "https://github.com/thatrandomasiandev/BPref.git"
REPO_BRANCH = "main"

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)
    ROOT = Path("/content/drive/MyDrive/LiraLab/BPref")
else:
    ROOT = Path("/content/BPref")

ROOT.parent.mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)

In [ ]:
# Cell 3 — Clone / update the experiment fork (includes pebble_reward_vs_rl)
import os
import subprocess
from pathlib import Path

def sh(cmd, cwd=None):
    print(">>", cmd)
    subprocess.check_call(cmd, shell=True, cwd=cwd)

if (ROOT / ".git").exists():
    sh(f"git fetch origin {REPO_BRANCH} && git checkout {REPO_BRANCH} && git pull --ff-only origin {REPO_BRANCH}", cwd=str(ROOT))
elif ROOT.exists() and any(ROOT.iterdir()):
    raise RuntimeError(f"{ROOT} exists but is not a git repo. Move/rename it, then re-run.")
else:
    sh(f"git clone --branch {REPO_BRANCH} --depth 1 {REPO_URL} {ROOT}")

assert (ROOT / "experiments/pebble_reward_vs_rl/CLAIM.md").exists(), (
    "CLAIM.md missing — wrong repo/branch. Need thatrandomasiandev/BPref with pebble_reward_vs_rl."
)
os.chdir(ROOT)
print("cwd:", Path.cwd())
print("commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
# Cell 4 — Dependencies (Colab already has CUDA PyTorch)
import os
import subprocess

os.environ["MUJOCO_GL"] = "egl"  # headless MuJoCo on Colab
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONPATH"] = f"{ROOT}:{ROOT / 'custom_dmc2gym'}"

pkgs = [
    "hydra-core",
    "omegaconf",
    "gym==0.26.2",
    "dm_control",
    "mujoco",
    "scikit-image",
    "tensorboard",
    "tqdm",
    "matplotlib",
]
subprocess.check_call(["pip", "install", "-q", *pkgs])
subprocess.check_call(["pip", "install", "-q", "-e", "custom_dmc2gym"], cwd=str(ROOT))

import torch
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("MUJOCO_GL", os.environ["MUJOCO_GL"])
print("PYTHONPATH", os.environ["PYTHONPATH"])

In [ ]:
# Cell 5 — Smoke (wiring only — NOT evidence)
import os
import subprocess
from pathlib import Path

os.chdir(ROOT)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONPATH"] = f"{ROOT}:{ROOT / 'custom_dmc2gym'}"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["CONDITION"] = "full"
os.environ["DEVICE"] = "cuda"
os.environ["SEED"] = "0"
os.environ["STEPS"] = "6000"
os.environ["ENV"] = "walker_walk"

smoke_dir = "experiments/pebble_reward_vs_rl/exp/walker_walk/full/seed0_colab_smoke"
cmd = [
    "bash",
    "experiments/pebble_reward_vs_rl/run_condition.sh",
    "num_train_steps=6000",
    "num_seed_steps=1000",
    "num_unsup_steps=2000",
    "num_interact=2000",
    "max_feedback=80",
    "reward_batch=20",
    "reward_update=5",
    "eval_frequency=4000",
    "num_eval_episodes=1",
    "diag_holdout_pairs=64",
    "diag_onpolicy_pairs=32",
    "diag_probe_gradient_steps=20",
    "diag_probe_steps=[6000]",
    "agent.batch_size=256",
    f"hydra.run.dir={smoke_dir}",
]
print("SMOKE (wiring only):", " ".join(cmd))
subprocess.check_call(cmd, cwd=str(ROOT))
csv = Path(smoke_dir) / "diagnostics.csv"
assert csv.exists(), csv
print("Smoke OK —", csv)
print("This is NOT a scientific finding.")

In [ ]:
# Cell 6 — Diagnostic suite (CLAIM.md §6: tentative patterns only)
# 5 seeds × 4 conditions × 100k on Walker-walk. Resume-safe (skips completed).
import os
import subprocess

os.chdir(ROOT)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYTHONPATH"] = f"{ROOT}:{ROOT / 'custom_dmc2gym'}"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["DEVICE"] = "cuda"
os.environ["PARALLEL"] = "1"  # one job per GPU — safest on a single A100
os.environ["STEPS"] = "100000"
os.environ["SEEDS"] = "1 2 3 4 5"
os.environ["ENV"] = "walker_walk"

print("Starting DIAGNOSTIC suite — tentative patterns only (not paper-scale).")
rc = subprocess.call(
    ["python", "experiments/pebble_reward_vs_rl/run_diagnostic_suite.py"],
    cwd=str(ROOT),
)
print("diagnostic suite exit:", rc)
if rc != 0:
    raise SystemExit(f"Diagnostic suite failed with rc={rc}. Check exp/_logs/.")

In [ ]:
# Cell 7 — Aggregate diagnostic results
import os
import subprocess
from pathlib import Path

os.chdir(ROOT)
subprocess.check_call(
    [
        "python",
        "experiments/pebble_reward_vs_rl/analyze_results.py",
        "--root",
        "experiments/pebble_reward_vs_rl/exp",
    ],
    cwd=str(ROOT),
)
summary = Path("experiments/pebble_reward_vs_rl/exp/_analysis/summary_by_condition.csv")
print(summary.read_text() if summary.exists() else "missing summary")
print("\nSTATUS: diagnostic aggregation only — not a justified H1/H2 conclusion.")

In [ ]:
# Cell 8 — Paper-scale suite (CLAIM.md claim gate)
# 10 seeds × 4 conditions × 500k + paper-ish reward hyperparameters.
# Expect many hours even on A100. Drive persistence strongly recommended.
import os
import subprocess

RUN_PAPER_SCALE = True  # set False to stop after diagnostic

if not RUN_PAPER_SCALE:
    print("Skipping paper-scale (RUN_PAPER_SCALE=False).")
else:
    os.chdir(ROOT)
    os.environ["MUJOCO_GL"] = "egl"
    os.environ["PYTHONPATH"] = f"{ROOT}:{ROOT / 'custom_dmc2gym'}"
    os.environ["PYTHONUNBUFFERED"] = "1"
    os.environ["DEVICE"] = "cuda"
    os.environ["PARALLEL"] = "1"
    os.environ["STEPS"] = "500000"
    os.environ["SEEDS"] = "1 2 3 4 5 6 7 8 9 10"
    os.environ["ENV"] = "walker_walk"
    print("Starting PAPER-SCALE suite — this is the claim gate.")
    rc = subprocess.call(
        ["python", "experiments/pebble_reward_vs_rl/run_paper_scale_suite.py"],
        cwd=str(ROOT),
    )
    print("paper-scale exit:", rc)
    if rc != 0:
        raise SystemExit(f"Paper-scale failed rc={rc}. Check exp/_logs/.")
    subprocess.check_call(
        [
            "python",
            "experiments/pebble_reward_vs_rl/analyze_results.py",
            "--root",
            "experiments/pebble_reward_vs_rl/exp",
        ],
        cwd=str(ROOT),
    )

## After runs

- Results live under `experiments/pebble_reward_vs_rl/exp/` (on Drive if `USE_DRIVE=True`).
- Fill `RESULTS.md` from `_analysis/summary_by_condition.csv` using CLAIM.md §7.
- **Do not** claim H1/H2 from smoke or incomplete seed sets.

If Colab disconnects mid-suite: reconnect the **same** runtime type, re-run cells 1–4, then re-run Cell 6 or 8 — suites skip completed `seed*` folders.